# 02 — pyrodigal Bacterial Host Genome Annotation

This notebook annotates bacterial (host) genomes using pyrodigal and produces INTERFACE-compliant outputs.

**References:**
- Hyatt D. et al. (2010) *Prodigal: prokaryotic gene recognition and translation initiation site identification.* BMC Bioinformatics 11:119. DOI: 10.1186/1471-2105-11-119
- Larralde M. (2022) *Pyrodigal: Python bindings and interface to Prodigal.* JOSS 7(72):4296. DOI: 10.21105/joss.04296

**Why pyrodigal (not PHANOTATE)?** Prodigal is trained on bacterial codon usage and ribosome binding sites; it assumes non-overlapping ORFs (correct for bacteria). PHANOTATE is optimised for phages (overlapping ORFs, different codon bias) and produces garbage on bacteria. pyrodigal is a Python binding to Prodigal with identical accuracy and ~2× faster runtime.

---

## 中文说明

本 Notebook 使用 pyrodigal 对细菌（宿主）基因组进行注释，输出符合 INTERFACE.md 规范的文件。

**为何选用 pyrodigal（而非 PHANOTATE）？** Prodigal 基于细菌密码子使用和核糖体结合位点训练，假设 ORF 不重叠（细菌情况正确）。PHANOTATE 针对噬菌体优化（重叠 ORF、不同密码子偏好），用于细菌会产生错误结果。pyrodigal 是 Prodigal 的 Python 绑定，精度相同，速度约快 2 倍。

**Output files / 输出文件:**
- `outputs/host_proteins/<acc>.faa` — protein FASTA with INTERFACE headers
- `outputs/host_orfs/<acc>.gff3`   — ORF coordinates in GFF3 format

In [ ]:
# Cell 2 — Imports and version printouts / 导入库并打印版本信息
import sys
from pathlib import Path

# Path anchoring / 路径锚定
NOTEBOOK_DIR = Path.cwd().resolve()
MODULE_ROOT  = NOTEBOOK_DIR.parent
REPO_ROOT    = MODULE_ROOT.parent
sys.path.insert(0, str(NOTEBOOK_DIR))

import pyrodigal
import Bio
from annotate_lib import PYRODIGAL_VERSION, run_prodigal, append_manifest, manifest_row_for_file

print(f"Python:         {sys.version}")
print(f"Biopython:      {Bio.__version__}")
print(f"pyrodigal:      {PYRODIGAL_VERSION}")
print(f"REPO_ROOT:      {REPO_ROOT}")

## Method: Prodigal gene prediction algorithm / 方法说明

Prodigal operates in two phases:
1. **Training** — builds a genome-specific model of codon usage, ribosome binding sites, and start codon frequencies from the input sequence.
2. **Prediction** — uses the trained model to score all ORFs and selects the optimal non-overlapping gene set.

We use `meta=False` (single-genome mode) so each bacterium gets its own trained model. This is appropriate for complete/draft genomes. Use `meta=True` only for metagenomic assemblies.

Prodigal 分两个阶段运行：
1. **训练** — 从输入序列构建基因组特异性密码子使用、核糖体结合位点和起始密码子频率模型。
2. **预测** — 用训练模型对所有 ORF 打分，选择最优的非重叠基因集。

使用 `meta=False`（单基因组模式），每个细菌获得独立训练的模型。仅对宏基因组装配使用 `meta=True`。

In [ ]:
# Cell 4 — Set output directory / 设置输出目录
OUTDIR = MODULE_ROOT / "outputs"
OUTDIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTDIR}")

## Sample run: Xanthomonas campestris pv. campestris str. 8004 (NZ_CP155948)

Tonight's sample run annotates **one Xcc strain** (NZ_CP155948.1 — str. 8004, the closest available relative of the reference Xcc ATCC 33913 that phiL7 infects). The full 34-bacterium batch is deferred to the Laguna HPC run.

今晚的样本运行仅注释**一株 Xcc**（NZ_CP155948.1 — str. 8004，与 phiL7 感染的参考菌株 Xcc ATCC 33913 最近的可用菌株）。对全部 34 株细菌的批量注释推迟到 Laguna HPC 运行。

**Sanity check:** Xcc ATCC 33913 has 4181 protein-coding genes (da Silva et al. 2002, Nature 417:459). NZ_CP155948 should be in a similar range.

**健全性检查：** Xcc ATCC 33913 有 4181 个蛋白质编码基因（da Silva et al. 2002，Nature 417:459）。NZ_CP155948 应在相近范围内。

In [ ]:
# Cell 8 — Sample run / 样本运行
BACTERIA_FNA = REPO_ROOT / "00_raw_data" / "bacteria" / "NZ_CP155948" / "genome.fna"
print(f"Input: {BACTERIA_FNA}")
assert BACTERIA_FNA.exists(), f"genome.fna not found: {BACTERIA_FNA}"

meta = run_prodigal(BACTERIA_FNA, OUTDIR)
print(f"\nResults for {meta['acc']}:")
print(f"  ORFs:         {meta['n_orfs']}")
print(f"  Mean len:     {meta['mean_orf_len']} aa")
print(f"  Runtime:      {meta['runtime_s']} s")
print(f"  FAA:          {meta['faa_path']}")
print(f"  GFF3:         {meta['gff3_path']}")

In [ ]:
# Cell 9 — Validation: header format + ORF count sanity check
# 验证：头格式 + ORF 数量健全性检查
import re
from pathlib import Path

HEADER_RE = re.compile(
    r'^>(\S+) \| source=(\S+) \| length=(\d+)'
    r' \| start=(\d+) \| end=(\d+) \| strand=([+-])'
    r' \| tool=(\S+)$'
)

faa_path = Path(meta['faa_path'])
headers  = [l for l in faa_path.read_text().splitlines() if l.startswith('>')]
bad      = [h for h in headers if not HEADER_RE.match(h)]
assert not bad, f"Malformed headers: {bad[:3]}"
print(f"All {len(headers)} headers pass INTERFACE regex ✓")

# da Silva 2002: ATCC 33913 has 4181 genes; str 8004 should be in [3000, 5500]
# da Silva 2002：ATCC 33913 有 4181 个基因；str 8004 应在 [3000, 5500] 范围内
assert 3000 <= len(headers) <= 5500, f"ORF count {len(headers)} outside expected range"
print(f"ORF count {len(headers)} in expected range [3000, 5500] ✓")

In [ ]:
# Cell 10 — Optional batch run (deferred to Laguna)
# 可选批量运行（推迟到 Laguna）
BATCH_ENABLED = False   # set True on Laguna / 在 Laguna 上设为 True

if BATCH_ENABLED:
    from concurrent.futures import ProcessPoolExecutor, as_completed
    bact_dirs = sorted((REPO_ROOT / "00_raw_data" / "bacteria").iterdir())
    batch_results = []
    with ProcessPoolExecutor(max_workers=4) as pool:
        futures = {
            pool.submit(run_prodigal, d / "genome.fna", OUTDIR): d.name
            for d in bact_dirs if (d / "genome.fna").exists()
        }
        for fut in as_completed(futures):
            try:
                batch_results.append(fut.result())
            except Exception as exc:
                print(f"FAILED {futures[fut]}: {exc}")
    print(f"Batch complete: {len(batch_results)} bacteria annotated")
else:
    print("Batch run skipped (BATCH_ENABLED=False). Set True for Laguna HPC run.")

In [ ]:
# Cell 11 — Append to MANIFEST.csv / 追加到 MANIFEST.csv
MANIFEST = OUTDIR / "MANIFEST.csv"
rows = [
    manifest_row_for_file(meta['faa_path'],  meta['n_orfs'], "pyrodigal protein FASTA"),
    manifest_row_for_file(meta['gff3_path'], meta['n_orfs'], "pyrodigal GFF3 ORF coords"),
]

import csv
existing_files = set()
if MANIFEST.exists():
    with open(MANIFEST) as f:
        existing_files = {r['filename'] for r in csv.DictReader(f)}
new_rows = [r for r in rows if r['filename'] not in existing_files]
if new_rows:
    append_manifest(MANIFEST, new_rows)
    print(f"Added {len(new_rows)} rows to MANIFEST.csv")
else:
    print("MANIFEST.csv already up to date.")